# Davinci Notebook 17 — Long-Context Curriculum & Synthetic Retrieval Tasks

Planning and evaluation utilities for gradually expanding context. Generates controlled synthetic retrieval examples and curriculum manifests; does not constitute long-context training or prove model capability. Synthetic tasks are for sanity checks and must not be the sole benchmark.

In [ ]:
import json, random, sys, subprocess
from pathlib import Path
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src' / 'davinci').is_dir():
    REPO_ROOT = Path('/content/Davinci')
    if not (REPO_ROOT / 'src' / 'davinci').is_dir():
        subprocess.run(['git', 'clone', 'https://github.com/keyhmoham/Davinci.git', str(REPO_ROOT)], check=True)
sys.path.insert(0, str(REPO_ROOT))
print('Davinci source:', REPO_ROOT)


## 1. Context-length curriculum
Progressively expand sequence length only after baseline quality and memory/throughput checks. Suggested stages are a conservative experiment plan, not guarantees.

In [ ]:
from src.davinci.long_context_eval import build_needle_example
CURRICULUM = [
    {'tokens': n, 'phase': 'base' if n == 4096 else ('experimental' if n == 500000 else 'extend'),
     'gate': 'Measure exact token length, memory, latency, and quality before advancing.'}
    for n in [4096, 8192, 16384, 32768, 65536, 131072, 262144, 500000]
]
print(json.dumps(CURRICULUM, indent=2))


## 2. Synthetic needle retrieval example generator
Produces examples with a unique key/value fact inserted in a distractor document. Use tokenized lengths to measure real context sizes; character count is not token count.

In [ ]:
from src.davinci.long_context_eval import build_needle_example
example = build_needle_example(3, documents=20, position_fraction=0.5, seed=42)
print(example['metadata'], example['answer'])


## 3. Build a stratified synthetic test set
Position and distractor count vary; persist seed and metadata for reproducibility.

In [ ]:
from src.davinci.long_context_eval import build_needle_example, measure_token_lengths, write_jsonl_and_manifest
import os
N_EXAMPLES = 100
examples = [
    build_needle_example(i, documents=40, position_fraction=fraction, seed=123 + i,
                         language='fa' if i % 2 else 'en')
    for i, fraction in enumerate([0.0, 0.25, 0.5, 0.75, 1.0] * (N_EXAMPLES // 5))
]
TOKENIZER_MODEL = Path('/content/drive/MyDrive/Davinci/tokenizer/davinci_sentencepiece.model')
if not TOKENIZER_MODEL.is_file():
    raise FileNotFoundError(f'Expected trained tokenizer at {TOKENIZER_MODEL}; train/export tokenizer first.')
try:
    import sentencepiece as spm
except ImportError as exc:
    raise RuntimeError('Install sentencepiece in this runtime before continuing.') from exc
sp = spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))
examples = measure_token_lengths(examples, lambda text: sp.encode(text, out_type=int))
print('examples:', len(examples), 'token length range:', min(x['metadata']['token_count'] for x in examples), max(x['metadata']['token_count'] for x in examples))


## 4. Scoring and manifest helpers

In [ ]:
from src.davinci.long_context_eval import score_retrieval, write_jsonl_and_manifest
OUTPUT = Path('/content/drive/MyDrive/Davinci/evaluation/long_context_synthetic.jsonl')
manifest = write_jsonl_and_manifest(examples, OUTPUT, tokenizer_id=str(TOKENIZER_MODEL), seed=123)
print(json.dumps(manifest, indent=2))
print('Saved synthetic evaluation set. Model inference is a separate step; no model score is fabricated.')


## 5. Long-context experiment gates

- Report exact token length, not character estimates.
- Measure evidence at beginning, middle, and end.
- Add distractors, multiple needles, cross-document reasoning, and retrieval failure cases.
- Compare against short-context baseline and shuffled evidence controls.
- Record KV-cache memory and prefill latency separately from decode latency.
- Verify model does not answer from memorized benchmark content.
- Do not claim 500K useful context from a configuration or positional extrapolation alone.